# Text2Swarm: Static 3D Point Cloud Generation & Sparse Downsampling Pipeline

A complete, end-to-end modular pipeline that converts natural language text descriptions into **sparse, highly recognizable 3D drone swarm coordinate formations** ($N \in [15, 200]$ drones).

### Pipeline Workflow:
1. **Hybrid Prompt Router & Generator:** Routes 3D semantic prompts to OpenAI Point-E diffusion, literal text (`text:WORD`) to 2D morphological edge extrusion, or procedural mocks for fast math testing (<0.01s).
2. **Pre-Cleaning & Normalization:** Statistical Outlier Removal (SOR) to filter disconnected "floater" points and bounding-box unit scaling ($[-1, 1]^3$).
3. **Geometry-Preserving Downsampling:**
   - **Mode A (Edge-Weighted 3D FPS):** Default for $N \ge 45$. Neighborhood covariance and eigenvalue decomposition prioritize high-curvature creases, wingtips, rims, and extremities before FPS.
   - **Mode B (2.5D PCA Silhouette + Spine):** Default for $N < 45$. Principal Component Analysis (SVD) and polar angular binning extract outer boundaries and primary spine features with $Z$-depth compression.
   - **Mode C (Uniform FPS):** Standard uniform surface baseline.
4. **Spatial Quality Metrics:** Computes minimum pairwise clearance ($d_{\min}$) and mean nearest-neighbor distance ($\bar{d}_{\text{nn}}$).
5. **Inline 3D Visualizers & Export:**
   - **True 3D Matplotlib Visualizer:** Direct inline 3D rendering (`Axes3D`) with depth shading and equal aspect ratio (guaranteed 100% visible in VS Code).
   - **Interactive Plotly 3D Visualizer:** Dual 3D scatter scenes with rotatable camera and coordinates hover inspector.
   - **File Exporters:** Saves `.npy` float32 coordinate array and structured `.json` metadata.

> **Scope Note:** This module strictly handles **static 3D coordinate generation and geometric preservation**. It does not perform trajectory path-planning, flight physics simulation, or Hungarian assignment.

## 1. Environment Setup & Hardware Auto-Detection

Safely detects GPU acceleration:
- If CUDA capability $\ge 7.0$ (e.g. Google Colab T4/V100/A100), enables CUDA acceleration.
- If GPU compute capability $< 7.0$ (e.g. local NVIDIA GeForce MX350 CC 6.1), automatically falls back to CPU to prevent PyTorch CUDA kernel errors.
- Sets `pio.renderers.default = "notebook+vscode"` to ensure 3D plots render directly inside VS Code.

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Hardware Auto-Detection
# ==============================================================================
import os
import sys
import re
import json
import time
from typing import Optional, Tuple, Dict, Any

import numpy as np
import scipy.ndimage as ndi
from scipy.spatial import cKDTree
from PIL import Image, ImageDraw, ImageFont

# Matplotlib 3D plotting
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Plotly 3D plotting
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Configure Plotly renderer for VS Code and Jupyter
pio.renderers.default = "notebook+vscode"

# PyTorch and Point-E
import torch
from point_e.diffusion.configs import DIFFUSION_CONFIGS, diffusion_from_config
from point_e.diffusion.sampler import PointCloudSampler
from point_e.models.download import load_checkpoint
from point_e.models.configs import MODEL_CONFIGS, model_from_config

def get_compute_device() -> torch.device:
    """
    Safely detects available compute hardware:
    - If CUDA is available and compute capability >= 7.0 (e.g. Colab T4/V100/A100), uses CUDA.
    - If CUDA capability < 7.0 (e.g. GeForce MX350 CC 6.1 on local laptop), gracefully falls back
      to CPU to prevent CUDA kernel incompatibilities.
    """
    if torch.cuda.is_available():
        try:
            cc_major, cc_minor = torch.cuda.get_device_capability(0)
            if cc_major >= 7:
                torch.zeros(1, device="cuda")
                print(f"CUDA GPU detected: {torch.cuda.get_device_name(0)} (Compute Capability {cc_major}.{cc_minor})")
                return torch.device("cuda")
            else:
                print(f"Detected GPU {torch.cuda.get_device_name(0)} (CC {cc_major}.{cc_minor} < 7.0). Falling back to CPU for stability.")
        except Exception as e:
            print(f"CUDA check failed ({e}). Falling back to CPU.")
    else:
        print("No CUDA GPU detected. Running on CPU.")
    return torch.device("cpu")

DEVICE = get_compute_device()
print(f"Active Compute Device: {DEVICE}")

## 2. OpenAI Point-E Model Loading & Singleton Cache

Loads the `base40M-textvec` checkpoint from the local directory cache (`./point_e_model_cache`). Caches the initialized sampler in memory so repeated runs never re-download or re-initialize weights.

In [ ]:
# ==============================================================================
# Cell 2: OpenAI Point-E Model Loading & Singleton Cache
# ==============================================================================
_CACHED_SAMPLER = None
_CACHED_DEVICE = None

def get_point_e_sampler(device: Optional[torch.device] = None) -> PointCloudSampler:
    """
    Loads and caches the Point-E base40M-textvec model in memory.
    Ensures repeated runs do not re-load model weights.
    """
    global _CACHED_SAMPLER, _CACHED_DEVICE
    if device is None:
        device = DEVICE

    if _CACHED_SAMPLER is not None and _CACHED_DEVICE == device:
        return _CACHED_SAMPLER

    print(f"[Point-E] Loading base40M-textvec model onto {device}...")
    base_name = 'base40M-textvec'
    base_model = model_from_config(MODEL_CONFIGS[base_name], device)
    base_model.eval()
    base_diffusion = diffusion_from_config(DIFFUSION_CONFIGS[base_name])

    local_cache = os.path.abspath("./point_e_model_cache")
    cache_dir = os.environ.get("POINT_E_CACHE_DIR", local_cache if os.path.isdir(local_cache) else None)
    base_model.load_state_dict(load_checkpoint(base_name, device, cache_dir=cache_dir))

    # Explicit 1-element lists for single base model
    sampler = PointCloudSampler(
        device=device,
        models=[base_model],
        diffusions=[base_diffusion],
        num_points=[1024],
        aux_channels=['R', 'G', 'B'],
        guidance_scale=[3.0],
        model_kwargs_key_filter=('texts',),
        use_karras=[True],
        karras_steps=[64],
        sigma_min=[1e-3],
        sigma_max=[120],
        s_churn=[3],
    )
    _CACHED_SAMPLER = sampler
    _CACHED_DEVICE = device
    print("[Point-E] Model loaded and cached successfully.")
    return _CACHED_SAMPLER

## 3. Statistical Outlier Removal & Unit Normalization

1. **Statistical Outlier Removal (SOR):** Computes mean distance to $k = 10$ neighbors for each point and strips detached floater points ($\mu_i > \mu_{\text{global}} + 2.0 \cdot \sigma_{\text{global}}$).
2. **Bounding-Box Normalization:** Centers coordinates at the origin and scales to $[-1.0, 1.0]^3$ while strictly preserving aspect ratio.

In [ ]:
# ==============================================================================
# Cell 3: Statistical Outlier Removal & Unit Normalization
# ==============================================================================
def normalize_point_cloud(coords: np.ndarray) -> np.ndarray:
    """
    Centers coords at the origin and scales uniformly to fit inside [-1.0, 1.0]^3.
    Preserves true geometric aspect ratio.
    """
    coords = coords.copy()
    centroid = np.mean(coords, axis=0)
    coords = coords - centroid
    max_dist = np.max(np.linalg.norm(coords, axis=1))
    coords = coords / (max_dist + 1e-8)
    return coords.astype(np.float32)

def remove_statistical_outliers(
    coords: np.ndarray,
    colors: Optional[np.ndarray] = None,
    k: int = 10,
    std_ratio: float = 2.0
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Statistical Outlier Removal (SOR) to eliminate detached floater points:
    1. Query k=10 nearest neighbors for each point via cKDTree.
    2. Compute mean distance mu_i to neighbors.
    3. Filter out points where mu_i > mu_global + std_ratio * sigma_global.
    """
    if len(coords) <= k:
        return coords, colors

    tree = cKDTree(coords)
    dists, _ = tree.query(coords, k=min(k + 1, len(coords)))
    mu_i = np.mean(dists[:, 1:], axis=1)

    mu_global = float(np.mean(mu_i))
    sigma_global = float(np.std(mu_i))
    threshold = mu_global + std_ratio * sigma_global

    keep_mask = mu_i <= threshold
    cleaned_coords = coords[keep_mask]
    cleaned_colors = colors[keep_mask] if colors is not None else None
    return cleaned_coords.astype(np.float32), cleaned_colors.astype(np.float32) if cleaned_colors is not None else None

## 4. Literal Text & 2D Symbol Generator (Contour Extrusion)

Handles literal words or flat icons (e.g. `"text:HELLO"`, `"text:DRONE"`). Renders high-contrast text, extracts morphological edges via binary erosion (`mask ^ binary_erosion(mask)` weighted $3\times$), and extrudes across 3 depth layers along $Z$.

In [ ]:
# ==============================================================================
# Cell 4: Literal Text & 2D Symbol Generator (Contour Extrusion)
# ==============================================================================
def generate_text_or_symbol_cloud(
    text: str,
    depth_layers: int = 3,
    z_range: Tuple[float, float] = (-0.08, 0.08),
    min_points: int = 1024
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Renders literal text or symbols onto a 2D canvas, extracts morphological edge perimeters
    (weighted 3x) and interior mask points, and extrudes across 3 depth layers along Z.
    """
    width, height = 600, 200
    img = Image.new('L', (width, height), color=0)
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.load_default(size=100)
    except TypeError:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    pos = ((width - text_w) // 2 - bbox[0], (height - text_h) // 2 - bbox[1])
    draw.text(pos, text, fill=255, font=font)

    mask = np.array(img) > 128
    if not np.any(mask):
        mask = np.ones((height, width), dtype=bool)

    eroded = ndi.binary_erosion(mask)
    edge_mask = mask ^ eroded
    interior_mask = eroded

    y_edge, x_edge = np.where(edge_mask)
    y_int, x_int = np.where(interior_mask)

    pts_edge = np.column_stack([x_edge.astype(np.float32), -y_edge.astype(np.float32)])
    pts_edge_weighted = np.tile(pts_edge, (3, 1))

    if len(y_int) > 0:
        stride = max(1, len(y_int) // 1000)
        pts_int = np.column_stack([x_int[::stride].astype(np.float32), -y_int[::stride].astype(np.float32)])
        pts_2d = np.vstack([pts_edge_weighted, pts_int])
    else:
        pts_2d = pts_edge_weighted

    z_layers = np.linspace(z_range[0], z_range[1], depth_layers)
    coords_layers = [np.column_stack([pts_2d, np.full(len(pts_2d), z)]) for z in z_layers]
    dense_3d = np.vstack(coords_layers)

    if len(dense_3d) < min_points:
        reps = int(np.ceil(min_points / len(dense_3d)))
        dense_3d = np.tile(dense_3d, (reps, 1))[:min_points]

    colors = np.tile([0.0, 0.85, 1.0], (len(dense_3d), 1)).astype(np.float32)
    return dense_3d.astype(np.float32), colors

## 5. Procedural Mock Aircraft Generator (Instant CPU Testing)

Generates a synthetic procedural 1,024-point 3D aircraft with fuselage cylinder, swept wings, vertical fin, and stabilizers in $<0.01$ seconds without loading PyTorch or Point-E weights. Enables instant testing of downsampling math and visualization.

In [ ]:
# ==============================================================================
# Cell 5: Procedural Mock Aircraft Generator (Instant CPU Testing)
# ==============================================================================
def generate_mock_aircraft_cloud(n_points: int = 1024) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generates a procedural, synthetic 1,024-point 3D aircraft with:
    - Cylindrical fuselage with tapered nose and tail (450 pts)
    - Cockpit windshield accent (cyan)
    - Swept main wings (420 pts, navy blue)
    - Swept vertical stabilizer / tail fin (100 pts, red)
    - Horizontal stabilizers (54 pts, blue)
    Runs in <0.01 seconds on CPU without loading weights.
    """
    pts = []
    colors = []

    # 1. Fuselage
    n_fuse = 450
    x_fuse = np.linspace(-0.75, 0.75, n_fuse)
    theta_fuse = np.linspace(0, 2 * np.pi, n_fuse, endpoint=False)
    r_fuse = 0.12 * np.ones(n_fuse)
    nose_mask = x_fuse > 0.30
    r_fuse[nose_mask] *= (0.75 - x_fuse[nose_mask]) / 0.45
    tail_mask = x_fuse < -0.30
    r_fuse[tail_mask] *= (x_fuse[tail_mask] + 0.75) / 0.45
    r_fuse = np.maximum(r_fuse, 0.015)

    y_fuse = r_fuse * np.cos(theta_fuse * 17)
    z_fuse = r_fuse * np.sin(theta_fuse * 17)

    fuse_pts = np.column_stack([x_fuse, y_fuse, z_fuse])
    fuse_colors = np.tile([0.85, 0.85, 0.90], (n_fuse, 1))
    cockpit_mask = (x_fuse > 0.20) & (x_fuse < 0.45) & (z_fuse > 0.03)
    fuse_colors[cockpit_mask] = [0.10, 0.75, 0.95]
    pts.append(fuse_pts)
    colors.append(fuse_colors)

    # 2. Swept Main Wings (210 per side)
    n_wings = 420
    n_half = n_wings // 2
    for side in [-1, 1]:
        span = np.linspace(0.10, 0.90, n_half)
        root_chord, tip_chord = 0.35, 0.10
        chord_len = root_chord - (root_chord - tip_chord) * (span - 0.10) / 0.80
        sweep_x = 0.15 - 0.50 * (span - 0.10) / 0.80

        rng = np.random.RandomState(42 if side == 1 else 43)
        s_along = rng.uniform(0.0, 1.0, n_half)
        wx = sweep_x - s_along * chord_len
        wy = side * span
        wz = 0.012 * np.sin(span * 3)

        wing_pts = np.column_stack([wx, wy, wz])
        wing_colors = np.tile([0.15, 0.25, 0.65], (n_half, 1))
        pts.append(wing_pts)
        colors.append(wing_colors)

    # 3. Vertical Tail Fin
    n_tail = 100
    z_tail = np.linspace(0.05, 0.45, n_tail)
    x_tail = -0.45 - 0.30 * (z_tail - 0.05) / 0.40 - np.linspace(0.0, 0.12, n_tail)
    y_tail = np.random.RandomState(44).normal(0.0, 0.005, n_tail)

    tail_pts = np.column_stack([x_tail, y_tail, z_tail])
    tail_colors = np.tile([0.90, 0.20, 0.20], (n_tail, 1))
    pts.append(tail_pts)
    colors.append(tail_colors)

    # 4. Horizontal Stabilizers
    n_hstab = 54
    n_h_half = n_hstab // 2
    for side in [-1, 1]:
        h_span = np.linspace(0.05, 0.35, n_h_half)
        hx = -0.55 - 0.15 * (h_span / 0.35)
        hy = side * h_span
        hz = np.full(n_h_half, 0.05)
        h_pts = np.column_stack([hx, hy, hz])
        h_colors = np.tile([0.20, 0.30, 0.70], (n_h_half, 1))
        pts.append(h_pts)
        colors.append(h_colors)

    all_pts = np.vstack(pts)[:n_points].astype(np.float32)
    all_colors = np.vstack(colors)[:n_points].astype(np.float32)
    return all_pts, all_colors

## 6. Hybrid Prompt Router (`generate_point_cloud`)

Intelligently routes generation:
- `mock=True` $\rightarrow$ Procedural mock aircraft
- `prompt.startswith("text:")` or `mode=="text"` $\rightarrow$ Extruded 2D text cloud
- Otherwise $\rightarrow$ OpenAI Point-E diffusion model + Statistical Outlier Removal

In [ ]:
# ==============================================================================
# Cell 6: Hybrid Prompt Router (generate_point_cloud)
# ==============================================================================
def generate_point_cloud(
    prompt: str,
    mock: bool = False,
    mode: str = "auto",
    device: Optional[torch.device] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Hybrid Prompt Router:
    - mock == True -> Procedural mock aircraft (1,024 points).
    - mode == "text" or prompt starts with "text:" -> Extruded 2D text cloud.
    - Otherwise -> Point-E 3D diffusion model + Statistical Outlier Removal.
    All clouds are centered and scaled via normalize_point_cloud.
    """
    clean_prompt = prompt.strip()
    if mock:
        print("[Generator] Generating procedural mock aircraft (1,024 points)...")
        coords, colors = generate_mock_aircraft_cloud(1024)
    elif mode == "text" or clean_prompt.lower().startswith("text:"):
        text_content = clean_prompt[5:].strip() if clean_prompt.lower().startswith("text:") else clean_prompt
        print(f"[Generator] Generating extruded 2D/3D text cloud for: '{text_content}'...")
        coords, colors = generate_text_or_symbol_cloud(text_content)
    else:
        print(f"[Generator] Sampling Point-E 3D diffusion model for: '{clean_prompt}'...")
        sampler = get_point_e_sampler(device=device)
        samples = None
        for x in sampler.sample_batch_progressive(batch_size=1, model_kwargs=dict(texts=[clean_prompt])):
            samples = x
        pc = sampler.output_to_point_clouds(samples)[0]
        raw_coords = pc.coords.astype(np.float32)
        raw_colors = np.stack([pc.channels['R'], pc.channels['G'], pc.channels['B']], axis=-1).astype(np.float32)

        print(f"[Generator] Point-E output: {len(raw_coords)} points. Applying Statistical Outlier Removal...")
        coords, colors = remove_statistical_outliers(raw_coords, raw_colors, k=10, std_ratio=2.0)
        print(f"[Generator] Outliers removed. Remaining points: {len(coords)}.")

    coords = normalize_point_cloud(coords)
    return coords.astype(np.float32), colors.astype(np.float32)

## 7. Sparse Point Cloud Reduction Algorithms

Implements three downsampling strategies:
1. **Core FPS ([`farthest_point_sampling`](#)):** Deterministic greedy distance maximization.
2. **Mode A ([`sample_edge_wireframe_3d`](#)):** 3D Edge-Weighted FPS ($N \ge 45$) using surface variation $\kappa$ and extremity $e$ to prioritize edges, wingtips, rims, and creases.
3. **Mode B ([`sample_silhouette_25d`](#)):** 2.5D PCA Silhouette + Spine ($N < 45$) using SVD projection, polar angular binning, and depth compression $Z \times 0.45$.
4. **Mode C ([`sample_uniform_fps`](#)):** Standard uniform surface baseline.

In [ ]:
# ==============================================================================
# Cell 7: Sparse Downsampling Algorithms (FPS, Mode A Edge3D, Mode B Silhouette25D)
# ==============================================================================
def farthest_point_sampling(points: np.ndarray, n_samples: int) -> np.ndarray:
    """
    Deterministically selects `n_samples` indices from `points` (M, 3)
    such that the minimum pairwise distance between selected points is maximized.
    """
    M = len(points)
    if n_samples >= M:
        return np.arange(M)

    selected_idx = np.zeros(n_samples, dtype=int)
    distances = np.full(M, np.inf)

    centroid = np.mean(points, axis=0)
    selected_idx[0] = np.argmax(np.linalg.norm(points - centroid, axis=1))

    for i in range(1, n_samples):
        last_pt = points[selected_idx[i - 1]]
        dist_to_last = np.sum((points - last_pt) ** 2, axis=1)
        distances = np.minimum(distances, dist_to_last)
        selected_idx[i] = np.argmax(distances)

    return selected_idx

def compute_surface_features(coords: np.ndarray, k: int = 20) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Computes local surface variation (curvature kappa) and extremity score (e):
    - Covariance matrix C_i of k nearest neighbors
    - Sorted eigenvalues lambda1 <= lambda2 <= lambda3
    - kappa = lambda1 / (sum lambda + 1e-8)
    - Priority score S = 0.70 * kappa_hat + 0.30 * e_hat
    """
    M = len(coords)
    k_eff = min(max(4, k), M)
    tree = cKDTree(coords)
    _, indices = tree.query(coords, k=k_eff)

    neighborhoods = coords[indices]
    means = np.mean(neighborhoods, axis=1, keepdims=True)
    diffs = neighborhoods - means
    covs = np.matmul(diffs.transpose(0, 2, 1), diffs) / float(k_eff)

    eigenvals = np.linalg.eigvalsh(covs)
    lambda1 = np.maximum(eigenvals[:, 0], 0.0)
    lambda2 = np.maximum(eigenvals[:, 1], 0.0)
    lambda3 = np.maximum(eigenvals[:, 2], 0.0)

    sum_lambdas = lambda1 + lambda2 + lambda3 + 1e-8
    kappa = lambda1 / sum_lambdas

    p_bar = np.mean(coords, axis=0)
    p_dists = np.linalg.norm(coords - p_bar, axis=1)
    e = p_dists / (np.max(p_dists) + 1e-8)

    kappa_min, kappa_max = np.min(kappa), np.max(kappa)
    kappa_hat = (kappa - kappa_min) / (kappa_max - kappa_min + 1e-8)

    e_min, e_max = np.min(e), np.max(e)
    e_hat = (e - e_min) / (e_max - e_min + 1e-8)

    S = 0.70 * kappa_hat + 0.30 * e_hat
    return kappa, e, S

def sample_edge_wireframe_3d(
    coords: np.ndarray,
    colors: Optional[np.ndarray],
    n_samples: int
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Mode A: 3D Wireframe Edge-Weighted FPS (Default for N >= 45).
    """
    M = len(coords)
    if n_samples >= M:
        return coords, colors

    _, _, S = compute_surface_features(coords, k=min(20, M))
    m_pool = max(int(np.floor(0.35 * M)), min(M, 2 * n_samples))
    candidate_indices = np.argsort(S)[::-1][:m_pool]
    candidate_coords = coords[candidate_indices]

    selected_local = farthest_point_sampling(candidate_coords, n_samples)
    final_indices = candidate_indices[selected_local]

    sampled_coords = coords[final_indices]
    sampled_colors = colors[final_indices] if colors is not None else None
    return sampled_coords.astype(np.float32), sampled_colors.astype(np.float32) if sampled_colors is not None else None

def sample_silhouette_25d(
    coords: np.ndarray,
    colors: Optional[np.ndarray],
    n_samples: int,
    align_to_camera: bool = False
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Mode B: 2.5D PCA Silhouette + Spine Sampling (Default for N < 45).
    """
    M = len(coords)
    if n_samples >= M:
        return coords, colors

    centroid = np.mean(coords, axis=0)
    centered = coords - centroid

    _, _, Vt = np.linalg.svd(centered, full_matrices=False)
    coords_pca = centered @ Vt.T

    X, Y = coords_pca[:, 0], coords_pca[:, 1]
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    B = max(72, 2 * n_samples)
    bin_edges = np.linspace(-np.pi, np.pi, B + 1)
    bin_idx = np.clip(np.digitize(theta, bin_edges) - 1, 0, B - 1)

    silhouette_indices = []
    for b in range(B):
        pts_in_bin = np.where(bin_idx == b)[0]
        if len(pts_in_bin) > 0:
            top_k = min(2, len(pts_in_bin))
            sorted_in_bin = pts_in_bin[np.argsort(r[pts_in_bin])[::-1][:top_k]]
            silhouette_indices.extend(sorted_in_bin)

    silhouette_indices = np.unique(silhouette_indices)

    kappa, _, _ = compute_surface_features(coords, k=min(20, M))
    num_spine = max(1, int(np.ceil(0.15 * M)))
    spine_indices = np.argsort(kappa)[::-1][:num_spine]

    pool_indices = np.unique(np.concatenate([silhouette_indices, spine_indices]))

    coords_pca_comp = coords_pca.copy()
    coords_pca_comp[:, 2] *= 0.45

    pool_pts = coords_pca_comp[pool_indices]
    selected_local = farthest_point_sampling(pool_pts, n_samples)
    final_indices = pool_indices[selected_local]

    if align_to_camera:
        sampled_coords = coords_pca_comp[final_indices]
    else:
        sampled_coords = coords_pca_comp[final_indices] @ Vt + centroid

    sampled_colors = colors[final_indices] if colors is not None else None
    return sampled_coords.astype(np.float32), sampled_colors.astype(np.float32) if sampled_colors is not None else None

def sample_uniform_fps(
    coords: np.ndarray,
    colors: Optional[np.ndarray],
    n_samples: int
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Mode C: Uniform FPS across all dense points.
    """
    selected_indices = farthest_point_sampling(coords, n_samples)
    sampled_coords = coords[selected_indices]
    sampled_colors = colors[selected_indices] if colors is not None else None
    return sampled_coords.astype(np.float32), sampled_colors.astype(np.float32) if sampled_colors is not None else None

def downsample_point_cloud(
    coords: np.ndarray,
    colors: Optional[np.ndarray] = None,
    num_points: int = 50,
    mode: str = "auto",
    align_to_camera: bool = False
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Main Downsampler Dispatcher.
    """
    resolved_mode = mode.lower()
    if resolved_mode == "auto":
        resolved_mode = "silhouette25d" if num_points < 45 else "edge3d"

    if resolved_mode == "edge3d":
        pts, cols = sample_edge_wireframe_3d(coords, colors, num_points)
    elif resolved_mode == "silhouette25d":
        pts, cols = sample_silhouette_25d(coords, colors, num_points, align_to_camera=align_to_camera)
    elif resolved_mode in ("uniform_fps", "uniform"):
        pts, cols = sample_uniform_fps(coords, colors, num_points)
    else:
        pts, cols = sample_edge_wireframe_3d(coords, colors, num_points)

    pts = normalize_point_cloud(pts)
    return pts.astype(np.float32), cols.astype(np.float32) if cols is not None else None

## 8. True 3D Visualizers (Matplotlib 3D & Plotly 3D), Metrics & Exporters

Includes:
- **`plot_3d_comparison_matplotlib(...)`:** Generates a **true 3D perspective scatter plot** (`Axes3D`) with depth shading and equal aspect ratio. Renders inline as a PNG, guaranteeing 100% visibility in VS Code without requiring WebGL or browser popups.
- **`plot_3d_gallery_matplotlib(...)`:** 1x3 true 3D side-by-side comparison of **Edge-3D vs. Silhouette-25D vs. Uniform-FPS**.
- **`create_comparison_plot(...)` & `show_figure(fig)`:** Interactive 3D Plotly WebGL viewer with rotatable camera.
- **`export_cloud(...)`:** Saves `.npy` float32 coordinate array and structured `.json` metadata.

In [ ]:
# ==============================================================================
# Cell 8: True 3D Visualizers (Matplotlib 3D & Plotly 3D), Metrics & Exporters
# ==============================================================================
def plot_3d_comparison_matplotlib(
    dense_coords: np.ndarray,
    dense_colors: Optional[np.ndarray],
    sparse_coords: np.ndarray,
    sparse_colors: Optional[np.ndarray],
    prompt: str = "",
    mode: str = ""
):
    """
    Renders a side-by-side TRUE 3D perspective plot using Matplotlib 3D (mplot3d).
    Guaranteed to render immediately inline in VS Code, Jupyter, and Colab without WebGL/browser issues.
    """
    metrics = compute_cloud_metrics(sparse_coords)
    N = metrics["num_points"]
    d_min = metrics["min_distance"]
    d_nn = metrics["mean_nn_distance"]

    fig = plt.figure(figsize=(16, 7))
    fig.patch.set_facecolor('#0b0f19')

    # --- Subplot 1: Dense 3D Cloud ---
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.set_facecolor('#0b0f19')
    c1 = dense_colors if dense_colors is not None else '#38bdf8'
    ax1.scatter(
        dense_coords[:, 0], dense_coords[:, 1], dense_coords[:, 2],
        c=c1, s=4, alpha=0.55, depthshade=True
    )
    ax1.set_title(f"Raw Dense 3D Cloud ({len(dense_coords)} Points)", color='white', fontsize=13, fontweight='bold', pad=12)
    ax1.set_xlabel("X (Width)", color='#94a3b8', labelpad=8)
    ax1.set_ylabel("Y (Depth)", color='#94a3b8', labelpad=8)
    ax1.set_zlabel("Z (Height)", color='#94a3b8', labelpad=8)
    ax1.tick_params(colors='#94a3b8', labelsize=8)
    ax1.view_init(elev=22, azim=45)
    ax1.set_box_aspect([1, 1, 1])

    # --- Subplot 2: Sparse 3D Swarm ---
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.set_facecolor('#0b0f19')
    c2 = sparse_colors if sparse_colors is not None else '#f43f5e'
    ax2.scatter(
        sparse_coords[:, 0], sparse_coords[:, 1], sparse_coords[:, 2],
        c=c2, s=70, edgecolors='white', linewidth=0.8, alpha=1.0, depthshade=True
    )
    # Labeled drone waypoints in 3D
    step = max(1, N // 8)
    for i in range(0, N, step):
        ax2.text(
            sparse_coords[i, 0] + 0.04,
            sparse_coords[i, 1] + 0.04,
            sparse_coords[i, 2] + 0.04,
            f"D{i+1}", color='#38bdf8', fontsize=8, fontweight='bold'
        )

    ax2.set_title(
        f"Sparse 3D Swarm ({N} Drones) | {mode.upper()}\n"
        f"d_min = {d_min:.3f} | d_nn = {d_nn:.3f}",
        color='white', fontsize=13, fontweight='bold', pad=12
    )
    ax2.set_xlabel("X (Width)", color='#94a3b8', labelpad=8)
    ax2.set_ylabel("Y (Depth)", color='#94a3b8', labelpad=8)
    ax2.set_zlabel("Z (Height)", color='#94a3b8', labelpad=8)
    ax2.tick_params(colors='#94a3b8', labelsize=8)
    ax2.view_init(elev=22, azim=45)
    ax2.set_box_aspect([1, 1, 1])

    # Stylize 3D panes with subtle grid
    for ax in [ax1, ax2]:
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor('#1e293b')
        ax.yaxis.pane.set_edgecolor('#1e293b')
        ax.zaxis.pane.set_edgecolor('#1e293b')
        ax.grid(color='#334155', linestyle=':', alpha=0.6)

    plt.suptitle(f"Text2Swarm 3D Point Cloud: '{prompt}'", color='white', fontsize=16, fontweight='bold', y=0.96)
    plt.tight_layout()
    plt.show()

def plot_3d_gallery_matplotlib(
    pts_edge: np.ndarray, cols_edge: Optional[np.ndarray],
    pts_sil: np.ndarray, cols_sil: Optional[np.ndarray],
    pts_uni: np.ndarray, cols_uni: Optional[np.ndarray],
    prompt: str = "",
    n_points: int = 40
):
    """
    Renders 1x3 TRUE 3D perspective subplots comparing Edge-3D, Silhouette-25D, and Uniform-FPS.
    """
    m_edge = compute_cloud_metrics(pts_edge)
    m_sil = compute_cloud_metrics(pts_sil)
    m_uni = compute_cloud_metrics(pts_uni)

    fig = plt.figure(figsize=(18, 6))
    fig.patch.set_facecolor('#0b0f19')

    configs = [
        (pts_edge, cols_edge, "#f43f5e", "Mode A: Edge-3D", m_edge),
        (pts_sil, cols_sil, "#06b6d4", "Mode B: 2.5D Silhouette+Spine", m_sil),
        (pts_uni, cols_uni, "#a855f7", "Mode C: Uniform FPS", m_uni)
    ]

    for idx, (pts, cols, default_col, name, m) in enumerate(configs, start=1):
        ax = fig.add_subplot(1, 3, idx, projection='3d')
        ax.set_facecolor('#0b0f19')
        c = cols if cols is not None else default_col
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=c, s=65, edgecolors='white', linewidth=0.7, depthshade=True)
        ax.set_title(
            f"<b>{name}</b>\n"
            f"d_min = {m['min_distance']:.3f} | d_nn = {m['mean_nn_distance']:.3f}",
            color='white', fontsize=12, pad=10
        )
        ax.set_xlabel("X", color='#94a3b8')
        ax.set_ylabel("Y", color='#94a3b8')
        ax.set_zlabel("Z", color='#94a3b8')
        ax.tick_params(colors='#94a3b8', labelsize=8)
        ax.view_init(elev=22, azim=45)
        ax.set_box_aspect([1, 1, 1])

        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor('#1e293b')
        ax.yaxis.pane.set_edgecolor('#1e293b')
        ax.zaxis.pane.set_edgecolor('#1e293b')
        ax.grid(color='#334155', linestyle=':', alpha=0.6)

    plt.suptitle(f"Text2Swarm 3D Strategy Comparison: '{prompt}' ({n_points} Points)", color='white', fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

def compute_cloud_metrics(coords: np.ndarray) -> Dict[str, float]:
    """
    Calculates spatial quality metrics:
    - num_points (N)
    - min_distance (d_min): minimum pairwise distance between any two points
    - mean_nn_distance (d_nn): average distance from each point to its nearest neighbor
    """
    N = len(coords)
    if N < 2:
        return {"num_points": N, "min_distance": 0.0, "mean_nn_distance": 0.0}

    diffs = coords[:, None, :] - coords[None, :, :]
    dists = np.linalg.norm(diffs, axis=-1)
    np.fill_diagonal(dists, np.inf)

    nearest_dists = np.min(dists, axis=1)
    min_dist = float(np.min(nearest_dists))
    mean_nn_dist = float(np.mean(nearest_dists))

    return {
        "num_points": N,
        "min_distance": round(min_dist, 4),
        "mean_nn_distance": round(mean_nn_dist, 4)
    }

def _format_plotly_colors(colors: Optional[np.ndarray], default_hex: str = "#00FFFF") -> Any:
    if colors is None or len(colors) == 0:
        return default_hex
    c_arr = np.array(colors)
    if np.max(c_arr) <= 1.0:
        c_arr = c_arr * 255.0
    c_arr = np.clip(c_arr, 0, 255).astype(int)
    return [f"rgb({r},{g},{b})" for r, g, b in c_arr]

def create_comparison_plot(
    dense_coords: np.ndarray,
    dense_colors: Optional[np.ndarray],
    sparse_coords: np.ndarray,
    sparse_colors: Optional[np.ndarray],
    prompt: str,
    mode: str,
    output_html: Optional[str] = None
) -> go.Figure:
    """
    1-row, 2-column interactive Plotly 3D comparison plot.
    """
    metrics = compute_cloud_metrics(sparse_coords)
    N = metrics["num_points"]
    d_min = metrics["min_distance"]
    d_nn = metrics["mean_nn_distance"]

    dense_color_vals = _format_plotly_colors(dense_colors, default_hex="#888888")
    sparse_color_vals = _format_plotly_colors(sparse_colors, default_hex="#FF3366")

    dense_hover = [f"Raw #{i}<br>X: {x:.2f}<br>Y: {y:.2f}<br>Z: {z:.2f}" for i, (x, y, z) in enumerate(dense_coords)]
    sparse_hover = [f"Drone #{i+1}<br>X: {x:.2f}<br>Y: {y:.2f}<br>Z: {z:.2f}" for i, (x, y, z) in enumerate(sparse_coords)]

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=(
            f"<b>Raw Dense Cloud ({len(dense_coords)} pts)</b>",
            f"<b>Sparse Swarm ({N} pts)</b> | Mode: {mode.upper()}"
        ),
        horizontal_spacing=0.04
    )

    fig.add_trace(
        go.Scatter3d(
            x=dense_coords[:, 0], y=dense_coords[:, 1], z=dense_coords[:, 2],
            mode="markers",
            marker=dict(size=2, color=dense_color_vals, opacity=0.5),
            hovertext=dense_hover, hoverinfo="text", name="Dense Cloud"
        ), row=1, col=1
    )

    fig.add_trace(
        go.Scatter3d(
            x=sparse_coords[:, 0], y=sparse_coords[:, 1], z=sparse_coords[:, 2],
            mode="markers",
            marker=dict(size=6, color=sparse_color_vals, opacity=1.0, line=dict(color="#FFFFFF", width=0.5)),
            hovertext=sparse_hover, hoverinfo="text", name=f"Sparse ({N} Drones)"
        ), row=1, col=2
    )

    scene_config = dict(
        xaxis=dict(title="X", backgroundcolor="#0b0f19", gridcolor="#1f293d", showbackground=True),
        yaxis=dict(title="Y", backgroundcolor="#0b0f19", gridcolor="#1f293d", showbackground=True),
        zaxis=dict(title="Z", backgroundcolor="#0b0f19", gridcolor="#1f293d", showbackground=True),
        aspectmode="data"
    )

    fig.update_layout(
        template="plotly_dark",
        title=dict(
            text=(
                f"<b>Text2Swarm: '{prompt}'</b><br>"
                f"<span style='font-size: 13px; color: #38bdf8;'>"
                f"Points: {N} &nbsp;|&nbsp; Min Pairwise Distance (d_min): {d_min:.3f} &nbsp;|&nbsp; "
                f"Mean NN Distance (d_nn): {d_nn:.3f} &nbsp;|&nbsp; Mode: {mode}"
                f"</span>"
            ),
            x=0.5, xanchor="center"
        ),
        scene=scene_config, scene2=scene_config,
        margin=dict(l=20, r=20, b=20, t=90),
        legend=dict(orientation="h", yanchor="bottom", y=-0.05, xanchor="center", x=0.5)
    )

    if output_html:
        os.makedirs(os.path.dirname(os.path.abspath(output_html)), exist_ok=True)
        fig.write_html(output_html)
        print(f"[Visualizer] Saved interactive 3D plot to: {output_html}")

    return fig

def show_figure(fig: go.Figure):
    """
    Renders Plotly figures inline reliably in VS Code, Jupyter, and Colab.
    """
    try:
        html_code = fig.to_html(include_plotlyjs='cdn', full_html=False)
        display(HTML(html_code))
    except Exception:
        try:
            fig.show(renderer="notebook+vscode")
        except Exception:
            fig.show()

def export_cloud(
    sparse_coords: np.ndarray,
    sparse_colors: Optional[np.ndarray],
    prompt: str,
    mode: str,
    output_dir: str = "./outputs",
    base_filename: Optional[str] = None
) -> Tuple[str, str]:
    """
    Exports .npy array (float32) and .json metadata file.
    """
    os.makedirs(output_dir, exist_ok=True)
    if base_filename is None:
        sanitized = re.sub(r'[^a-zA-Z0-9_\-]+', '_', prompt.strip()).strip('_') or "swarm"
        base_filename = f"{sanitized}_{len(sparse_coords)}pts_{mode}"

    npy_path = os.path.join(output_dir, f"{base_filename}.npy")
    json_path = os.path.join(output_dir, f"{base_filename}.json")

    coords_f32 = sparse_coords.astype(np.float32)
    np.save(npy_path, coords_f32)

    metrics = compute_cloud_metrics(sparse_coords)
    points_list = coords_f32.tolist()

    if sparse_colors is not None:
        colors_arr = np.array(sparse_colors)
        if np.max(colors_arr) > 1.0:
            colors_arr = colors_arr / 255.0
        colors_list = colors_arr.tolist()
    else:
        colors_list = [[1.0, 0.2, 0.4] for _ in range(len(sparse_coords))]

    payload = {
        "prompt": prompt,
        "mode": mode,
        "num_points": int(metrics["num_points"]),
        "min_distance": float(metrics["min_distance"]),
        "mean_nn_distance": float(metrics["mean_nn_distance"]),
        "points": points_list,
        "colors": colors_list
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print(f"[Export] Saved .npy coordinate array to: {npy_path}")
    print(f"[Export] Saved .json metadata and points to: {json_path}")
    return npy_path, json_path

## 9. Interactive Pipeline Execution & Inline 3D Visualization

Configure the parameters below and run the cell. The cell generates:
1. **True 3D Matplotlib Plot:** Side-by-side 3D perspective views of the raw dense 3D cloud vs. the sparse 3D drone swarm.
2. **Interactive 3D Plotly Subplot:** Fully rotatable 3D scene.
3. **Saved Output Files:** Paths to `.npy`, `.json`, and standalone `.html`.

In [ ]:
# ==============================================================================
# Cell 9: Interactive Pipeline Execution & Inline 3D Visualization
# ==============================================================================
# --- User Configuration ---
PROMPT = "fighter jet"     # e.g. "fighter jet", "an apple", or "text:SWARM"
NUM_POINTS = 35            # Target number of drones N (typically 15 - 200)
MODE = "auto"              # 'auto', 'edge3d', 'silhouette25d', 'uniform_fps', or 'text'
USE_MOCK = True            # Set True for instant <0.01s testing, False for real Point-E diffusion
OUTPUT_DIR = "./outputs"

# 1. Resolve Mode
resolved_mode = MODE.lower()
if resolved_mode == "auto":
    if PROMPT.strip().lower().startswith("text:"):
        resolved_mode = "silhouette25d"
    elif NUM_POINTS < 45:
        resolved_mode = "silhouette25d"
    else:
        resolved_mode = "edge3d"

print("=" * 65)
print(f"Generating Swarm: '{PROMPT}' | Target Points: {NUM_POINTS} | Mode: {resolved_mode} | Mock: {USE_MOCK}")
print("=" * 65)

# 2. Generate Dense Cloud
t0 = time.time()
dense_coords, dense_colors = generate_point_cloud(
    prompt=PROMPT,
    mock=USE_MOCK,
    mode=MODE,
    device=DEVICE
)
t_gen = time.time() - t0

# 3. Downsample to Sparse Swarm
t1 = time.time()
sparse_coords, sparse_colors = downsample_point_cloud(
    coords=dense_coords,
    colors=dense_colors,
    num_points=NUM_POINTS,
    mode=resolved_mode
)
t_down = time.time() - t1

# 4. Metrics & File Export
metrics = compute_cloud_metrics(sparse_coords)
print(f"Dense Generation: {len(dense_coords)} pts in {t_gen:.2f}s | Downsampling: {len(sparse_coords)} pts in {t_down:.3f}s")
print(f"Quality Metrics:  Points = {metrics['num_points']} | d_min = {metrics['min_distance']:.4f} | d_nn = {metrics['mean_nn_distance']:.4f}")

sanitized = re.sub(r'[^a-zA-Z0-9_\-]+', '_', PROMPT.strip()).strip('_') or "swarm"
base_name = f"{sanitized}_{NUM_POINTS}pts_{resolved_mode}"
html_file = os.path.join(OUTPUT_DIR, f"{base_name}_comparison.html")

export_cloud(sparse_coords, sparse_colors, PROMPT, resolved_mode, OUTPUT_DIR, base_name)

# 5. Render TRUE 3D Plotly & Matplotlib Figures
# A) True 3D Matplotlib Plot (100% visible in VS Code)
plot_3d_comparison_matplotlib(
    dense_coords, dense_colors,
    sparse_coords, sparse_colors,
    prompt=PROMPT, mode=resolved_mode
)

# B) Interactive 3D Plotly Figure
fig = create_comparison_plot(
    dense_coords, dense_colors,
    sparse_coords, sparse_colors,
    prompt=PROMPT, mode=resolved_mode,
    output_html=html_file
)
show_figure(fig)

# C) Clickable link to standalone HTML
display(HTML(f"""
<div style="margin-top:10px; padding:12px; background-color:#1e293b; border-radius:8px; border-left:4px solid #38bdf8;">
    <span style="color:#94a3b8; font-size:12px;">INTERACTIVE 3D STANDALONE FILE:</span><br>
    <a style="color:#38bdf8; font-weight:bold; font-size:14px; text-decoration:underline;" href="{html_file}" target="_blank">
        Click here to open {os.path.basename(html_file)} in full screen browser
    </a>
</div>
"""))

## 10. Comparative Gallery: Edge-3D vs. Silhouette-25D vs. Uniform FPS

Compares all three downsampling algorithms side-by-side on the exact same 3D model in **true 3D perspective** to visually verify feature priority, contour definition, and uniform spacing.

In [ ]:
# ==============================================================================
# Cell 10: Comparative Analysis: Edge-3D vs. Silhouette-25D vs. Uniform FPS
# ==============================================================================
COMPARE_PROMPT = "fighter jet"
N_COMPARE = 40

print(f"Running comparative analysis on '{COMPARE_PROMPT}' with N={N_COMPARE} points...")
base_coords, base_colors = generate_point_cloud(prompt=COMPARE_PROMPT, mock=True)

# 1. Edge-Weighted 3D
pts_edge, cols_edge = downsample_point_cloud(base_coords, base_colors, num_points=N_COMPARE, mode="edge3d")

# 2. 2.5D PCA Silhouette + Spine
pts_sil, cols_sil = downsample_point_cloud(base_coords, base_colors, num_points=N_COMPARE, mode="silhouette25d")

# 3. Uniform FPS
pts_uni, cols_uni = downsample_point_cloud(base_coords, base_colors, num_points=N_COMPARE, mode="uniform_fps")

# A) True 3D Matplotlib Gallery (100% visible in VS Code)
plot_3d_gallery_matplotlib(
    pts_edge, cols_edge,
    pts_sil, cols_sil,
    pts_uni, cols_uni,
    prompt=COMPARE_PROMPT,
    n_points=N_COMPARE
)

# B) Interactive 3D Plotly Comparison
m_edge = compute_cloud_metrics(pts_edge)
m_sil = compute_cloud_metrics(pts_sil)
m_uni = compute_cloud_metrics(pts_uni)

fig_comp = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=(
        f"<b>Mode A: Edge-3D</b><br>d_min={m_edge['min_distance']:.3f} | d_nn={m_edge['mean_nn_distance']:.3f}",
        f"<b>Mode B: 2.5D Silhouette+Spine</b><br>d_min={m_sil['min_distance']:.3f} | d_nn={m_sil['mean_nn_distance']:.3f}",
        f"<b>Mode C: Uniform FPS</b><br>d_min={m_uni['min_distance']:.3f} | d_nn={m_uni['mean_nn_distance']:.3f}"
    ),
    horizontal_spacing=0.03
)

for col_idx, (pts, cols, name, color_hex) in enumerate([
    (pts_edge, cols_edge, "Edge-3D", "#f43f5e"),
    (pts_sil, cols_sil, "Silhouette-25D", "#06b6d4"),
    (pts_uni, cols_uni, "Uniform-FPS", "#a855f7")
], start=1):
    fig_comp.add_trace(
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode="markers",
            marker=dict(size=6, color=_format_plotly_colors(cols, default_hex=color_hex), opacity=1.0, line=dict(color="#ffffff", width=0.5)),
            name=name
        ),
        row=1, col=col_idx
    )

scene_cfg = dict(
    xaxis=dict(title="X", backgroundcolor="#0b0f19", gridcolor="#1f293d", showbackground=True),
    yaxis=dict(title="Y", backgroundcolor="#0b0f19", gridcolor="#1f293d", showbackground=True),
    zaxis=dict(title="Z", backgroundcolor="#0b0f19", gridcolor="#1f293d", showbackground=True),
    aspectmode="data"
)
fig_comp.update_layout(
    template="plotly_dark",
    title=dict(
        text=f"<b>Downsampling Strategy Comparison for '{COMPARE_PROMPT}' ({N_COMPARE} Points)</b>",
        x=0.5, xanchor="center"
    ),
    scene=scene_cfg, scene2=scene_cfg, scene3=scene_cfg,
    margin=dict(l=10, r=10, b=20, t=80)
)
show_figure(fig_comp)